# Sequential curling final-score models

Configure and run the sequential experiment from `sequential_training.py`.

In [1]:
import importlib
import sys
import json
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import data_generation
import stats
from scratch import sequential_training as seq
importlib.reload(seq)
importlib.reload(stats)

<module 'stats' from '/home/vietafan/curling/stats.py'>

In [2]:
N = 5
rows_per_model = 10_000
state_generator_version = "turn_based"
# seq.show_state_generator_versions()
# Keep large datasets and model checkpoints outside the repository.
output_root = Path("/home/vietafan/curling_data")
resume_from = None
training_stages = 1
policy_comparison_num_sims = 5
run_dir = output_root / f"N{N}_rows{rows_per_model}_stages{training_stages}"
artifact_dir = run_dir / "sequential_experiment"
diagnostic_output_dir = run_dir / "sequential_diagnostics"  # or None to disable
experiment = seq.make_experiment_setup(state_generator_version)


In [3]:
run = seq.train_sequential_models(
    N=N, rows_per_model=rows_per_model, artifact_dir=artifact_dir,
    resume_from=resume_from, training_stages=training_stages,
    experiment=experiment,
    diagnostic_output_dir=diagnostic_output_dir,
    policy_comparison_num_sims=policy_comparison_num_sims,
)
models = run.models
datasets = run.datasets
validation_datasets = run.validation_datasets
training_info = run.training_info


KeyboardInterrupt: 

In [ ]:
# Standard validation statistics for each intermediate model.
for i, (model, normalizer) in models.items():
    print(f'Round {i}')
    model_stats = seq.evaluate_model(
        model, normalizer, validation_datasets[i], N=N
    )
    stats.print_stats(model_stats)
    diagnostic_records = []
    if diagnostic_output_dir is not None and diagnostic_output_dir.exists():
        for path in diagnostic_output_dir.glob(f'm_{i}_stats.jsonl'):
            diagnostic_records.extend(
                json.loads(line) for line in path.read_text().splitlines()
            )
    stats.plot_calibration_and_training_losses(
        model_stats, training_info[i]['train_loss'],
        training_info[i]['validation_loss'],
        diagnostic_records=diagnostic_records, title=f'Round {i}'
    )


In [ ]:
# Final model data uses only the rounds trained above: 2N-1 down to
# 2N-training_stages.
first_trained_round = 2 * N - training_stages
trained_rounds = range(first_trained_round, 2 * N)
final_data = seq.generate_final_dataset(
    models, k=first_trained_round,
    fractions={i: 1 / training_stages for i in trained_rounds},
    num_rows=rows_per_model, N=N, seed=123,
    random_sheet_states=experiment.random_sheet_states,
)
final_model, final_normalizer, final_info, final_validation_data = seq.train_model(
    final_data, seed=123, max_stones=2 * N, num_stones_per_side=N
)
seq.write_model(artifact_dir / 'final_model.npz', final_model, final_normalizer)
seq.save_metadata(artifact_dir / 'metadata.json', {
    'N': N, 'rows_per_model': rows_per_model, 'training_stages': training_stages,
    'trained_rounds': list(trained_rounds),
    'feature_width': seq.feature_width(2 * N),
    'training_info': training_info, 'final_training_info': final_info,
})
